In [ ]:
#| default_exp models

# models

> Pydantic data models shared across the entire pipeline.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field

## Story Analysis Models

In [ ]:
#| export
class CharacterArc(BaseModel):
    """Describes a character's state in a particular phase of the story."""
    phase: str
    description: str

In [ ]:
#| export
class Character(BaseModel):
    """A character in the story, with stable visual and narrative properties."""
    name: str
    aliases: list[str] = Field(default_factory=list)
    physical_description: str
    personality: str
    arcs: list[CharacterArc] = Field(default_factory=list)
    reference_image_prompt: str = ""
    """Stable prompt fragment used in every panel this character appears in."""
    reference_image_path: Path | None = None
    """Path to a generated character sheet image (set during render phase if supported)."""

In [ ]:
#| export
class Location(BaseModel):
    """A recurring location in the story."""
    name: str
    description: str
    visual_prompt: str
    """Stable prompt fragment reused in every panel set at this location."""
    atmosphere: str = ""
    """Mood cues: lighting, colour palette, era."""

In [ ]:
#| export
class StoryAnalysis(BaseModel):
    """Output of the analysis step: extracted characters, locations and themes."""
    title: str
    synopsis: str
    characters: list[Character]
    locations: list[Location]
    themes: list[str] = Field(default_factory=list)
    source_chunks: list[str] = Field(default_factory=list)
    """The chunked story text that produced this analysis (for traceability)."""

## Storyboard Models

In [ ]:
#| export
class DialogueBubble(BaseModel):
    """A single speech/thought/caption bubble in a panel."""
    speaker: str
    """Character name, or 'narration' for non-character text. Always exactly ONE name."""
    text: str
    bubble_type: Literal["speech", "shout", "whisper", "thought", "caption", "sfx"] = "speech"
    """
    speech  — normal spoken dialogue (oval bubble, pointed tail)
    shout   — yelling or strong emphasis (jagged spiky starburst bubble)
    whisper — quiet or secret speech (small oval, dashed border)
    thought — internal monologue (cloud shape, dotted trail)
    caption — narrator or scene text (rectangular box at panel edge)
    sfx     — sound effects like BANG, CRASH (large bold stylized text)
    """

In [ ]:
#| export
class Panel(BaseModel):
    """A single comic panel."""
    panel_number: int
    scene_id: str
    characters_present: list[str] = Field(default_factory=list)
    location: str
    action_description: str
    visual_prompt: str
    """Fully assembled image-gen prompt for this panel."""
    dialogue: list[DialogueBubble] = Field(default_factory=list)
    mood: str = ""
    camera_angle: str = ""
    """E.g. 'close-up', 'wide shot', 'over-the-shoulder'."""

In [ ]:
#| export
class Scene(BaseModel):
    """A group of panels covering one continuous story beat."""
    scene_id: str
    title: str
    source_chunk: str
    """The story excerpt this scene covers."""
    panels: list[Panel] = Field(default_factory=list)

In [ ]:
#| export
class Storyboard(BaseModel):
    """The complete comic playbook: all scenes and panels."""
    title: str
    scenes: list[Scene] = Field(default_factory=list)
    total_panels: int = 0

    @property
    def all_panels(self) -> list[Panel]:
        return [p for s in self.scenes for p in s.panels]

## Validation & Output Models

In [ ]:
#| export
class ValidationResult(BaseModel):
    """Result of the optional storyboard validation step."""
    passed: bool
    coverage_score: float
    """0.0–1.0. How well the storyboard covers the source story."""
    missing_story_beats: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)

In [ ]:
#| export
class RenderResult(BaseModel):
    """Result of rendering a single panel."""
    panel_number: int
    image_path: Path
    backend_used: str
    prompt_used: str
    metadata: dict = Field(default_factory=dict)

In [ ]:
#| export
class ComicOutput(BaseModel):
    """Final output of a complete pipeline run."""
    output_dir: Path
    analysis_path: Path
    storyboard_path: Path
    validation_path: Path | None = None
    rendered_panels: list[RenderResult] = Field(default_factory=list)
    html_path: Path | None = None

## Basic Tests

In [ ]:
# Verify models instantiate and serialise correctly
char = Character(
    name="Wei Chen",
    physical_description="Tall young man with short black hair",
    personality="Determined and loyal",
    reference_image_prompt="young Chinese man, short black hair, determined expression",
)
assert char.name == "Wei Chen"
assert char.aliases == []

storyboard = Storyboard(title="Test", scenes=[
    Scene(scene_id="s1", title="Opening", source_chunk="Once upon a time...", panels=[
        Panel(panel_number=1, scene_id="s1", location="Forest",
              action_description="Hero walks in", visual_prompt="hero walks through forest")
    ])
])
assert len(storyboard.all_panels) == 1
print("Models OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()